# Loan Application Process Analysis
This notebook performs a basic analysis of the loan application log from the BPI Challenge 2012 dataset.

In [ ]:

import xml.etree.ElementTree as ET
from datetime import datetime
from collections import defaultdict

# Helper function to parse the XES log using only the standard library

def parse_xes(filepath):
    ns = {'xes': 'http://www.xes-standard.org/'}
    root = ET.parse(filepath).getroot()
    for trace in root.findall('xes:trace', ns):
        attrs = {}
        events = []
        for child in trace:
            tag = child.tag.split('}')[-1]
            if tag == 'event':
                ev = {c.attrib.get('key'): c.attrib.get('value') for c in child if 'key' in c.attrib}
                events.append(ev)
            else:
                key = child.attrib.get('key')
                if key:
                    attrs[key] = child.attrib.get('value')
        yield {'attrs': attrs, 'events': events}


In [ ]:

# Filtering of events: keep only COMPLETE transitions and remove consecutive duplicates

def clean_events(events):
    filtered = [ev for ev in events if ev.get('lifecycle:transition') == 'COMPLETE']
    result = []
    prev = None
    for ev in filtered:
        name = ev.get('concept:name')
        if name != prev:
            result.append(ev)
        prev = name
    return result

def case_metrics(case):
    events = clean_events(case['events'])
    if not events:
        return None
    times = [datetime.fromisoformat(ev['time:timestamp']) for ev in events]
    duration = (max(times) - min(times)).total_seconds() / 3600.0
    size = len(events)
    unique_names = {ev['concept:name'] for ev in events}
    repeats = size - len(unique_names)
    outcome = events[-1]['concept:name']
    return {
        'case_id': case['attrs'].get('concept:name'),
        'amount_req': float(case['attrs'].get('AMOUNT_REQ', 0)),
        'duration_hours': duration,
        'size': size,
        'repeats': repeats,
        'outcome': outcome
    }


In [ ]:

log_path = 'financial_log.xes'
all_cases = list(parse_xes(log_path))
metrics = [case_metrics(c) for c in all_cases]
metrics = [m for m in metrics if m and m['outcome'] in {'A_APPROVED','A_DECLINED','A_CANCELLED'}]

segments = defaultdict(list)
for m in metrics:
    segments[m['outcome']].append(m)
print('Total analysed cases:', len(metrics))


In [ ]:

for outcome, items in segments.items():
    durations = [i['duration_hours'] for i in items]
    sizes = [i['size'] for i in items]
    amounts = [i['amount_req'] for i in items]
    repeats = [i['repeats'] for i in items]
    def avg(lst):
        return sum(lst)/len(lst) if lst else 0
    print('
===', outcome, '===')
    print('cases:', len(items))
    print('avg duration (h):', round(avg(durations),2))
    print('avg size:', round(avg(sizes),2))
    print('avg amount:', round(avg(amounts),2))
    print('avg repeated activities:', round(avg(repeats),2))


The above statistics can be used to assess bottlenecks and differences between cases ending with approval, decline or cancellation. Higher durations or repeated activities may hint at opportunities for automation or step reordering.